In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd

root = Path.cwd()
while not (root / "data" / "db").exists() and root != root.parent:
    root = root.parent

con = sqlite3.connect(root / "data" / "db" / "tennis.db")
games = pd.read_sql("""
    SELECT match_id, file_year, tourney_start, tourney_level,
           winner_id, loser_id, winner_rank, loser_rank
    FROM matches
    ORDER BY match_id
""", con)
games["day"] = (pd.to_datetime(games["tourney_start"]) - pd.Timestamp("2000-01-01")).dt.days


def run_elo(games, start=1450, k_new=40, k_old=20, n_new=30, level_mult=None,
            gap_days=None, gap_mult=1.0, win_bonus=0.0, scale=400,
            upset_cap=None, reset_days=None, reset_frac=1.0):
    level_mult = level_mult or {}
    rating, played, last, out = {}, {}, {}, []
    for r in games.itertuples(index=False):
        w, l = r.winner_id, r.loser_id
        rw, rl = rating.get(w, start), rating.get(l, start)
        nw, nl = played.get(w, 0), played.get(l, 0)
        gw = r.day - last.get(w, r.day)
        gl = r.day - last.get(l, r.day)

        if reset_days is not None:
            if gw > reset_days:
                rw = start + (1 - reset_frac) * (rw - start)
            if gl > reset_days:
                rl = start + (1 - reset_frac) * (rl - start)

        m = level_mult.get(r.tourney_level, 1.0)
        exp_w = 1 / (1 + 10 ** ((rl - rw) / scale))

        surprise = 1 - exp_w
        if upset_cap is not None:
            surprise = min(surprise, upset_cap)

        kw = (k_new if nw < n_new else k_old) * (gap_mult if gap_days is not None and gw > gap_days else 1.0)
        kl = (k_new if nl < n_new else k_old) * (gap_mult if gap_days is not None and gl > gap_days else 1.0)
        bonus = win_bonus if gap_days is not None and gw > gap_days else 0.0

        new_w = rw + m * kw * surprise + bonus
        new_l = rl - m * kl * surprise
        rating[w], rating[l] = new_w, new_l
        played[w], played[l] = nw + 1, nl + 1
        last[w], last[l] = r.day, r.day

        out.append((r.match_id, r.file_year, r.tourney_start, w, l, rw, rl, new_w, new_l, exp_w))
    res = pd.DataFrame(out, columns=["match_id", "file_year", "tourney_start", "winner_id", "loser_id",
                                     "r_winner", "r_loser", "r_winner_after", "r_loser_after", "exp_w"])
    return res, rating, played


def score(res, y0, y1):
    d = res[(res["file_year"] >= y0) & (res["file_year"] <= y1)]
    p = d["exp_w"]
    return {
        "years": f"{y0}-{y1}",
        "matches": len(d),
        "accuracy_%": round(100 * ((p > 0.5).mean() + 0.5 * (p == 0.5).mean()), 1),
        "log_loss": round(float(-np.log(p).mean()), 4),
        "brier": round(float(((1 - p) ** 2).mean()), 4),
    }


print("games:", games.shape, "| run_elo defined:", "run_elo" in globals())

In [ ]:
root = Path.cwd()
while not (root / "data" / "db").exists() and root != root.parent:
    root = root.parent

con = sqlite3.connect(root / "data" / "db" / "tennis.db")
games = pd.read_sql("""
    SELECT match_id, file_year, tourney_start, tourney_level,
           winner_id, loser_id, winner_rank, loser_rank
    FROM matches
    ORDER BY match_id
""", con)
games["day"] = (pd.to_datetime(games["tourney_start"]) - pd.Timestamp("2000-01-01")).dt.days
print(games.shape)

In [ ]:
res, final_rating, final_played = run_elo(games)

a = res[["match_id", "tourney_start", "winner_id", "loser_id", "r_winner", "r_winner_after", "exp_w"]].copy()
a.columns = ["match_id", "tourney_start", "player_id", "opponent_id", "elo_before", "elo_after", "expected"]
a["result"] = 1

b = res[["match_id", "tourney_start", "loser_id", "winner_id", "r_loser", "r_loser_after", "exp_w"]].copy()
b.columns = ["match_id", "tourney_start", "player_id", "opponent_id", "elo_before", "elo_after", "expected"]
b["expected"] = 1 - b["expected"]
b["result"] = 0

elo_long = pd.concat([a, b]).sort_values(["match_id", "result"], ascending=[True, False]).reset_index(drop=True)
print(elo_long.shape)


In [ ]:
n = pd.read_sql("""
    SELECT winner_id AS player_id, winner_name AS name, match_id FROM matches
    UNION ALL
    SELECT loser_id, loser_name, match_id FROM matches
""", con)
players = n.sort_values("match_id").drop_duplicates("player_id", keep="last")[["player_id", "name"]]

elo_long["source"] = "Calculated from TennisMyLife match data"
elo_long["licence"] = "Derived from TML data; non-commercial, see data/Sources.md"
elo_long["method"] = "Elo v1: start 1450, K=40 first 30 matches then 20, scale 400, overall rating"
players["source"] = "TennisMyLife match files (names as given)"
players["licence"] = "Derived from TML data; non-commercial, see data/Sources.md"

elo_long.to_sql("elo_long", con, if_exists="replace", index=False)
players.to_sql("players", con, if_exists="replace", index=False)
con.execute("CREATE INDEX IF NOT EXISTS idx_elo_player ON elo_long(player_id, match_id)")
con.commit()

print(pd.read_sql("""
    SELECT (SELECT COUNT(*) FROM matches) AS matches,
           (SELECT COUNT(*) FROM elo_long) / 2 AS elo_matches,
           (SELECT MAX(match_id) FROM matches) AS max_id_matches,
           (SELECT MAX(match_id) FROM elo_long) AS max_id_elo
""", con).to_string(index=False))
print(len(players), "players")

In [ ]:
def score(res, y0, y1):
    d = res[(res["file_year"] >= y0) & (res["file_year"] <= y1)]
    p = d["exp_w"]
    return {
        "years": f"{y0}-{y1}",
        "matches": len(d),
        "accuracy_%": round(100 * ((p > 0.5).mean() + 0.5 * (p == 0.5).mean()), 1),
        "log_loss": round(float(-np.log(p).mean()), 4),
        "brier": round(float(((1 - p) ** 2).mean()), 4),
    }

res, final_rating, final_played = run_elo(games)
print(pd.DataFrame([score(res, 2005, 2018), score(res, 2019, 2026)]).to_string(index=False))

g = games[games["winner_rank"].notna() & games["loser_rank"].notna() & (games["winner_rank"] != games["loser_rank"])]
for y0, y1 in [(2005, 2018), (2019, 2026)]:
    d = g[(g["file_year"] >= y0) & (g["file_year"] <= y1)]
    print(f"ATP ranking baseline {y0}-{y1}: {round(100 * (d['winner_rank'] < d['loser_rank']).mean(), 1)}%")

In [ ]:
test, _, _ = run_elo(games.head(1000))
print(test.shape)
print(test.head(3)[["match_id", "r_winner", "r_loser", "exp_w"]].round(3).to_string(index=False))

In [ ]:
res, final_rating, final_played = run_elo(games)
print(res.shape)

In [ ]:
def score(res, y0, y1):
    d = res[(res["file_year"] >= y0) & (res["file_year"] <= y1)]
    p = d["exp_w"]
    return {
        "years": f"{y0}-{y1}",
        "matches": len(d),
        "accuracy_%": round(100 * ((p > 0.5).mean() + 0.5 * (p == 0.5).mean()), 1),
        "log_loss": round(float(-np.log(p).mean()), 4),
        "brier": round(float(((1 - p) ** 2).mean()), 4),
    }

print(pd.DataFrame([score(res, 2005, 2018), score(res, 2019, 2026)]).to_string(index=False))

g = games[games["winner_rank"].notna() & games["loser_rank"].notna() & (games["winner_rank"] != games["loser_rank"])]
for y0, y1 in [(2005, 2018), (2019, 2026)]:
    d = g[(g["file_year"] >= y0) & (g["file_year"] <= y1)]
    print(f"ATP ranking baseline {y0}-{y1}: {round(100 * (d['winner_rank'] < d['loser_rank']).mean(), 1)}%")

In [ ]:
m = res.merge(games[["match_id", "winner_rank", "loser_rank"]], on="match_id")
m = m[m["winner_rank"].notna() & m["loser_rank"].notna()
      & (m["winner_rank"] != m["loser_rank"]) & (m["exp_w"] != 0.5)].copy()
m["elo_ok"] = (m["exp_w"] > 0.5).astype(int)
m["rank_ok"] = (m["winner_rank"] < m["loser_rank"]).astype(int)

for y0, y1 in [(2005, 2018), (2019, 2026)]:
    d = m[(m["file_year"] >= y0) & (m["file_year"] <= y1)]
    diff = d["elo_ok"] - d["rank_ok"]
    se = diff.std() / np.sqrt(len(d))
    print(f"{y0}-{y1}: n={len(d)}  Elo {100*d['elo_ok'].mean():.1f}%  ATP rank {100*d['rank_ok'].mean():.1f}%  "
          f"difference {100*diff.mean():.2f} points (95% interval {100*(diff.mean()-1.96*se):.2f} to {100*(diff.mean()+1.96*se):.2f})")

In [ ]:
d = res[(res["file_year"] >= 2005) & (res["file_year"] <= 2018) & (res["exp_w"] != 0.5)].copy()
d["p_fav"] = np.maximum(d["exp_w"], 1 - d["exp_w"])
d["fav_won"] = (d["exp_w"] > 0.5).astype(int)
d["bin"] = pd.cut(d["p_fav"], [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 1.0])

t = d.groupby("bin", observed=True).agg(
    matches=("fav_won", "size"),
    predicted_pct=("p_fav", lambda s: round(100 * s.mean(), 1)),
    actual_pct=("fav_won", lambda s: round(100 * s.mean(), 1)),
)
print(t.to_string())

In [ ]:
rows = []
for k_old in [10, 15, 20, 25, 30]:
    for k_new in [20, 30, 40]:
        r_, _, _ = run_elo(games, k_new=k_new, k_old=k_old)
        s = score(r_, 2005, 2018)
        rows.append({"k_old": k_old, "k_new": k_new, "accuracy_%": s["accuracy_%"], "log_loss": s["log_loss"], "brier": s["brier"]})

grid = pd.DataFrame(rows).sort_values("log_loss")
print(grid.to_string(index=False))

In [ ]:
#weight by tournament
variants = {
    "baseline (no weights)": {},
    "slams x1.25": {"G": 1.25},
    "slams 1.25, masters/finals 1.1": {"G": 1.25, "M": 1.1, "F": 1.1},
    "slams 1.5, masters/finals 1.25": {"G": 1.5, "M": 1.25, "F": 1.25},
    "250s x0.8": {"250": 0.8},
    "gradient": {"G": 1.3, "M": 1.15, "F": 1.15, "500": 1.0, "250": 0.85},
}

rows = []
for name, mult in variants.items():
    r_, _, _ = run_elo(games, level_mult=mult)
    s = score(r_, 2005, 2018)
    rows.append({"variant": name, "accuracy_%": s["accuracy_%"], "log_loss": s["log_loss"], "brier": s["brier"]})

print(pd.DataFrame(rows).sort_values("log_loss").to_string(index=False))

In [ ]:
variants2 = {
    "baseline": {},
    "gap 180d, K x1.5": dict(gap_days=180, gap_mult=1.5),
    "gap 365d, K x1.5": dict(gap_days=365, gap_mult=1.5),
    "gap 180d, +5 win bonus": dict(gap_days=180, win_bonus=5.0),
}

rows = []
for name, kw in variants2.items():
    r_, _, _ = run_elo(games, **kw)
    s = score(r_, 2005, 2018)
    rows.append({"variant": name, "accuracy_%": s["accuracy_%"], "log_loss": s["log_loss"], "brier": s["brier"]})

print(pd.DataFrame(rows).sort_values("log_loss").to_string(index=False))

In [ ]:
rows = []
for scale in [400, 450, 500, 600, 700]:
    r_, _, _ = run_elo(games, scale=scale)
    s = score(r_, 2005, 2018)
    rows.append({"scale": scale, "accuracy_%": s["accuracy_%"], "log_loss": s["log_loss"], "brier": s["brier"]})

print(pd.DataFrame(rows).sort_values("log_loss").to_string(index=False))

In [ ]:
rows = []
for scale in [300, 350, 400]:
    r_, _, _ = run_elo(games, scale=scale)
    s = score(r_, 2005, 2018)
    rows.append({"scale": scale, "accuracy_%": s["accuracy_%"], "log_loss": s["log_loss"], "brier": s["brier"]})

print(pd.DataFrame(rows).sort_values("log_loss").to_string(index=False))

In [ ]:
res, final_rating, final_played = run_elo(games)

a = res[["match_id", "tourney_start", "winner_id", "loser_id", "r_winner", "r_winner_after", "exp_w"]].copy()
a.columns = ["match_id", "tourney_start", "player_id", "opponent_id", "elo_before", "elo_after", "expected"]
a["result"] = 1

b = res[["match_id", "tourney_start", "loser_id", "winner_id", "r_loser", "r_loser_after", "exp_w"]].copy()
b.columns = ["match_id", "tourney_start", "player_id", "opponent_id", "elo_before", "elo_after", "expected"]
b["expected"] = 1 - b["expected"]
b["result"] = 0

elo_long = pd.concat([a, b]).sort_values(["match_id", "result"], ascending=[True, False]).reset_index(drop=True)
print(elo_long.shape)
print(elo_long.head(4).round(1).to_string(index=False))

In [ ]:
n = pd.read_sql("""
    SELECT winner_id AS player_id, winner_name AS name, match_id FROM matches
    UNION ALL
    SELECT loser_id, loser_name, match_id FROM matches
""", con)
players = n.sort_values("match_id").drop_duplicates("player_id", keep="last")[["player_id", "name"]]

elo_long["source"] = "Calculated from TennisMyLife match data"
elo_long["licence"] = "Derived from TML data; non-commercial, see data/SOURCES.md"
elo_long["method"] = "Elo v1: start 1450, K=40 first 30 matches then 20, scale 400, overall rating"
players["source"] = "TennisMyLife match files (names as given)"
players["licence"] = "Derived from TML data; non-commercial, see data/SOURCES.md"

elo_long.to_sql("elo_long", con, if_exists="replace", index=False)
players.to_sql("players", con, if_exists="replace", index=False)
con.execute("CREATE INDEX IF NOT EXISTS idx_elo_player ON elo_long(player_id, match_id)")
con.commit()

print(len(players), "players")
print(pd.read_sql("SELECT COUNT(*) AS rows_total FROM elo_long", con).to_string(index=False))

In [ ]:
variants3 = {
    "baseline": {},
    "upset cap 0.7": dict(upset_cap=0.7),
    "upset cap 0.6": dict(upset_cap=0.6),
    "upset cap 0.5": dict(upset_cap=0.5),
    "reset to 1450 after 182 days": dict(reset_days=182, reset_frac=1.0),
    "halfway reset after 182 days": dict(reset_days=182, reset_frac=0.5),
}

rows = []
for name, kw in variants3.items():
    r_, _, _ = run_elo(games, **kw)
    s = score(r_, 2005, 2018)
    rows.append({"variant": name, "accuracy_%": s["accuracy_%"], "log_loss": s["log_loss"], "brier": s["brier"]})

print(pd.DataFrame(rows).sort_values("log_loss").to_string(index=False))

In [ ]:
def snapshot(as_of, min_matches=15, min_recent=0, max_gap_days=182, top=20):
    since = (pd.Timestamp(as_of) - pd.Timedelta(days=364)).strftime("%Y-%m-%d")
    cutoff = (pd.Timestamp(as_of) - pd.Timedelta(days=max_gap_days)).strftime("%Y-%m-%d")
    q = """
    WITH upto AS (
        SELECT * FROM elo_long WHERE tourney_start <= :as_of
    ),
    last_row AS (
        SELECT player_id, MAX(match_id) AS last_match
        FROM upto GROUP BY player_id
    ),
    stats AS (
        SELECT player_id,
               COUNT(*) AS matches_total,
               SUM(CASE WHEN tourney_start > :since THEN 1 ELSE 0 END) AS matches_52w,
               MAX(tourney_start) AS last_start
        FROM upto GROUP BY player_id
    )
    SELECT p.name, ROUND(u.elo_after) AS elo, s.matches_52w, s.matches_total, s.last_start,
           CASE WHEN s.matches_total < 30 THEN 'provisional' ELSE '' END AS note
    FROM last_row l
    JOIN upto u ON u.player_id = l.player_id AND u.match_id = l.last_match
    JOIN stats s ON s.player_id = l.player_id
    JOIN players p ON p.player_id = l.player_id
    WHERE s.matches_total >= :min_matches
      AND s.matches_52w >= :min_recent
      AND s.last_start >= :cutoff
    ORDER BY u.elo_after DESC
    LIMIT :top
    """
    df = pd.read_sql(q, con, params={"as_of": as_of, "since": since, "cutoff": cutoff,
                                     "min_matches": min_matches, "min_recent": min_recent, "top": top})
    df.insert(0, "rank", range(1, len(df) + 1))
    return df

print(snapshot("2024-12-31").to_string(index=False))

In [ ]:
print(pd.read_sql("""
    SELECT DISTINCT file_year, tourney_name, tourney_start
    FROM matches
    WHERE tourney_start >= '2024-12-27' AND tourney_start <= '2025-01-05'
    ORDER BY tourney_start
""", con).to_string(index=False))

In [ ]:
print(snapshot("2024-12-22", min_recent=15).to_string(index=False))
print(snapshot("2024-12-22", min_recent=10).to_string(index=False))

In [ ]:
def period_view(start, end, min_matches=15, min_in_period=1, max_gap_days=182, top=20):
    el = pd.read_sql("SELECT match_id, tourney_start, player_id, elo_after FROM elo_long ORDER BY match_id", con)
    names = pd.read_sql("SELECT player_id, name FROM players", con).set_index("player_id")["name"]

    upto = el[el["tourney_start"] <= end]
    inside = upto[upto["tourney_start"] >= start]
    before = upto[upto["tourney_start"] < start].groupby("player_id")["elo_after"].last()

    g = inside.groupby("player_id").agg(elo_end=("elo_after", "last"),
                                        peak=("elo_after", "max"),
                                        matches_in_period=("match_id", "count"),
                                        last_start=("tourney_start", "max"))
    g["elo_start"] = before.reindex(g.index)
    g["change"] = g["elo_end"] - g["elo_start"]
    g["matches_total"] = upto.groupby("player_id").size().reindex(g.index)

    cutoff = (pd.Timestamp(end) - pd.Timedelta(days=max_gap_days)).strftime("%Y-%m-%d")
    g = g[(g["matches_total"] >= min_matches) & (g["matches_in_period"] >= min_in_period)
          & (g["last_start"] >= cutoff)]
    g = g.sort_values("elo_end", ascending=False).head(top)
    g = g.join(names)
    g.insert(0, "rank", range(1, len(g) + 1))
    return g[["rank", "name", "elo_start", "elo_end", "change", "peak", "matches_in_period"]].round(0)

In [ ]:
print(period_view("2024-01-01", "2024-12-22", min_in_period=15).head(10).to_string(index=False))

In [ ]:
games_s = pd.read_sql("""
    SELECT match_id, file_year, tourney_start, winner_id, loser_id, surface
    FROM matches
    ORDER BY match_id
""", con)
print(games_s.shape)

In [ ]:
def run_elo_surface(games, start=1450, k_new=40, k_old=20, n_new=30, scale=400, k_surf=10):
    rating, off, played, out = {}, {}, {}, []
    for r in games.itertuples(index=False):
        w, l, s = r.winner_id, r.loser_id, r.surface
        rw, rl = rating.get(w, start), rating.get(l, start)
        ow, ol = off.get((w, s), 0.0), off.get((l, s), 0.0)
        nw, nl = played.get(w, 0), played.get(l, 0)

        exp_w = 1 / (1 + 10 ** (((rl + ol) - (rw + ow)) / scale))
        surprise = 1 - exp_w

        kw = k_new if nw < n_new else k_old
        kl = k_new if nl < n_new else k_old
        rating[w], rating[l] = rw + kw * surprise, rl - kl * surprise
        off[(w, s)], off[(l, s)] = ow + k_surf * surprise, ol - k_surf * surprise
        played[w], played[l] = nw + 1, nl + 1

        out.append((r.match_id, r.file_year, r.tourney_start, w, l, exp_w))
    return pd.DataFrame(out, columns=["match_id", "file_year", "tourney_start", "winner_id", "loser_id", "exp_w"])

In [ ]:
rows = []
for k in [0, 5, 10, 20, 30]:
    r_ = run_elo_surface(games_s, k_surf=k)
    s_ = score(r_, 2005, 2018)
    rows.append({"k_surf": k, "accuracy_%": s_["accuracy_%"], "log_loss": s_["log_loss"], "brier": s_["brier"]})

print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
a = run_elo_surface(games_s, k_surf=0)
b = run_elo_surface(games_s, k_surf=10)

mask = ((a["file_year"] >= 2005) & (a["file_year"] <= 2018)).to_numpy()
diff = (-np.log(a["exp_w"].to_numpy()[mask])) - (-np.log(b["exp_w"].to_numpy()[mask]))
se = diff.std(ddof=1) / np.sqrt(len(diff))
print(f"2005-2018: log-loss improvement {diff.mean():.5f}  (95% interval {diff.mean() - 1.96 * se:.5f} to {diff.mean() + 1.96 * se:.5f})")

In [ ]:
print(pd.DataFrame([{"model": "Elo v1", **score(a, 2019, 2026)},
                    {"model": "surface Elo, k_surf=10", **score(b, 2019, 2026)}]).to_string(index=False))

mask2 = (a["file_year"] >= 2019).to_numpy()
diff2 = (-np.log(a["exp_w"].to_numpy()[mask2])) - (-np.log(b["exp_w"].to_numpy()[mask2]))
se2 = diff2.std(ddof=1) / np.sqrt(len(diff2))
print(f"2019-2026: log-loss improvement {diff2.mean():.5f}  (95% interval {diff2.mean() - 1.96 * se2:.5f} to {diff2.mean() + 1.96 * se2:.5f})")

In [ ]:
d = b[(b["file_year"] >= 2005) & (b["file_year"] <= 2018) & (b["exp_w"] != 0.5)].copy()
d["p_fav"] = np.maximum(d["exp_w"], 1 - d["exp_w"])
d["fav_won"] = (d["exp_w"] > 0.5).astype(int)
d["bin"] = pd.cut(d["p_fav"], [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 1.0])

t = d.groupby("bin", observed=True).agg(
    matches=("fav_won", "size"),
    predicted_pct=("p_fav", lambda s_: round(100 * s_.mean(), 1)),
    actual_pct=("fav_won", lambda s_: round(100 * s_.mean(), 1)),
)
print(t.to_string())

In [ ]:
v2 = run_elo_surface(games_s, k_surf=10)
x = np.log(v2["exp_w"] / (1 - v2["exp_w"])).to_numpy()
tr = ((v2["file_year"] >= 2005) & (v2["file_year"] <= 2018)).to_numpy()
te = (v2["file_year"] >= 2019).to_numpy()

slope = 1.0
for _ in range(25):
    s = 1 / (1 + np.exp(-slope * x[tr]))
    grad = (x[tr] * (1 - s)).sum()
    hess = -(x[tr] ** 2 * s * (1 - s)).sum()
    slope = slope - grad / hess
print("fitted slope:", round(float(slope), 3))

p_old = v2["exp_w"].to_numpy()[te]
p_new = 1 / (1 + np.exp(-slope * x[te]))
l_old, l_new = -np.log(p_old), -np.log(p_new)
diff = l_old - l_new
se = diff.std(ddof=1) / np.sqrt(len(diff))
print(f"2019-2026 log loss: v2 {l_old.mean():.4f} -> calibrated {l_new.mean():.4f}")
print(f"improvement {diff.mean():.5f}  (95% interval {diff.mean() - 1.96 * se:.5f} to {diff.mean() + 1.96 * se:.5f})")

In [ ]:
def run_elo_v2(games, start=1450, k_new=40, k_old=20, n_new=30, scale=400, k_surf=10):
    rating, off, played, out = {}, {}, {}, []
    for r in games.itertuples(index=False):
        w, l, s = r.winner_id, r.loser_id, r.surface
        rw, rl = rating.get(w, start), rating.get(l, start)
        ow, ol = off.get((w, s), 0.0), off.get((l, s), 0.0)
        nw, nl = played.get(w, 0), played.get(l, 0)

        exp_w = 1 / (1 + 10 ** (((rl + ol) - (rw + ow)) / scale))
        surprise = 1 - exp_w

        kw = k_new if nw < n_new else k_old
        kl = k_new if nl < n_new else k_old
        new_rw, new_rl = rw + kw * surprise, rl - kl * surprise
        new_ow, new_ol = ow + k_surf * surprise, ol - k_surf * surprise
        rating[w], rating[l] = new_rw, new_rl
        off[(w, s)], off[(l, s)] = new_ow, new_ol
        played[w], played[l] = nw + 1, nl + 1

        out.append((r.match_id, r.file_year, r.tourney_start, w, l, s,
                    rw, rl, new_rw, new_rl, ow, ol, new_ow, new_ol, exp_w))
    cols = ["match_id", "file_year", "tourney_start", "winner_id", "loser_id", "surface",
            "r_w", "r_l", "r_w_after", "r_l_after", "o_w", "o_l", "o_w_after", "o_l_after", "exp_raw"]
    return pd.DataFrame(out, columns=cols)

In [ ]:
v2 = run_elo_v2(games_s)
x = np.log(v2["exp_raw"] / (1 - v2["exp_raw"])).to_numpy()
tr = ((v2["file_year"] >= 2005) & (v2["file_year"] <= 2018)).to_numpy()

slope = 1.0
for _ in range(25):
    s_ = 1 / (1 + np.exp(-slope * x[tr]))
    slope = slope - (x[tr] * (1 - s_)).sum() / (-(x[tr] ** 2 * s_ * (1 - s_)).sum())
v2["exp_cal"] = 1 / (1 + np.exp(-slope * x))
print("calibration slope:", round(float(slope), 3))

raw = v2.rename(columns={"exp_raw": "exp_w"})
cal = v2.rename(columns={"exp_cal": "exp_w"})
print(pd.DataFrame([
    {"model": "v2 raw", **score(raw, 2005, 2018)},
    {"model": "v2 raw", **score(raw, 2019, 2026)},
    {"model": "v2 calibrated", **score(cal, 2019, 2026)},
]).to_string(index=False))

In [ ]:
cur = con.execute("SELECT method FROM elo_long LIMIT 1").fetchone()
if cur is not None and str(cur[0]).startswith("Elo v1"):
    con.execute("DROP TABLE IF EXISTS elo_long_v1")
    con.execute("CREATE TABLE elo_long_v1 AS SELECT * FROM elo_long")

wcols = ["match_id", "tourney_start", "winner_id", "loser_id", "surface",
         "r_w", "r_w_after", "o_w", "o_w_after", "exp_raw", "exp_cal"]
a = v2[wcols].copy()
a.columns = ["match_id", "tourney_start", "player_id", "opponent_id", "surface",
             "elo_before", "elo_after", "surf_offset_before", "surf_offset_after", "expected_raw", "expected"]
a["result"] = 1

lcols = ["match_id", "tourney_start", "loser_id", "winner_id", "surface",
         "r_l", "r_l_after", "o_l", "o_l_after", "exp_raw", "exp_cal"]
b = v2[lcols].copy()
b.columns = a.columns[:-1]
b["expected_raw"] = 1 - b["expected_raw"]
b["expected"] = 1 - b["expected"]
b["result"] = 0

elo_v2 = pd.concat([a, b]).sort_values(["match_id", "result"], ascending=[True, False]).reset_index(drop=True)
elo_v2["source"] = "Calculated from TennisMyLife match data"
elo_v2["licence"] = "Derived from TML data; non-commercial, see data/Sources.md"
elo_v2["method"] = (f"Elo v2: overall rating + surface offset (k_surf=10), start 1450, K=40 first 30 matches then 20, "
                    f"scale 400; probabilities calibrated with slope {slope:.3f} fitted on 2005-2018")

elo_v2.to_sql("elo_long", con, if_exists="replace", index=False)
con.execute("CREATE INDEX IF NOT EXISTS idx_elo_player ON elo_long(player_id, match_id)")
con.commit()

print(pd.read_sql("""
    SELECT (SELECT COUNT(*) FROM elo_long) / 2 AS v2_matches,
           (SELECT COUNT(*) FROM elo_long_v1) / 2 AS v1_matches,
           (SELECT COUNT(*) FROM matches) AS matches
""", con).to_string(index=False))

In [ ]:
print(snapshot("2024-12-22", min_recent=15).to_string(index=False))

In [ ]:
def snapshot_surface(as_of, surface, min_matches=15, min_recent=15, min_surface=10, max_gap_days=182, top=20):
    since = (pd.Timestamp(as_of) - pd.Timedelta(days=364)).strftime("%Y-%m-%d")
    cutoff = (pd.Timestamp(as_of) - pd.Timedelta(days=max_gap_days)).strftime("%Y-%m-%d")
    q = """
    WITH upto AS (
        SELECT * FROM elo_long WHERE tourney_start <= :as_of
    ),
    last_any AS (
        SELECT player_id, MAX(match_id) AS mid FROM upto GROUP BY player_id
    ),
    last_surf AS (
        SELECT player_id, MAX(match_id) AS mid FROM upto WHERE surface = :surface GROUP BY player_id
    ),
    stats AS (
        SELECT player_id,
               COUNT(*) AS matches_total,
               SUM(CASE WHEN tourney_start > :since THEN 1 ELSE 0 END) AS matches_52w,
               SUM(CASE WHEN surface = :surface THEN 1 ELSE 0 END) AS surface_matches,
               MAX(tourney_start) AS last_start
        FROM upto GROUP BY player_id
    )
    SELECT p.name,
           ROUND(a.elo_after) AS overall,
           ROUND(COALESCE(sf.surf_offset_after, 0)) AS surf_offset,
           ROUND(a.elo_after + COALESCE(sf.surf_offset_after, 0)) AS elo_surface,
           s.matches_52w, s.surface_matches
    FROM last_any la
    JOIN upto a ON a.player_id = la.player_id AND a.match_id = la.mid
    LEFT JOIN last_surf ls ON ls.player_id = la.player_id
    LEFT JOIN upto sf ON sf.player_id = ls.player_id AND sf.match_id = ls.mid
    JOIN stats s ON s.player_id = la.player_id
    JOIN players p ON p.player_id = la.player_id
    WHERE s.matches_total >= :min_matches AND s.matches_52w >= :min_recent
      AND s.surface_matches >= :min_surface AND s.last_start >= :cutoff
    ORDER BY a.elo_after + COALESCE(sf.surf_offset_after, 0) DESC
    LIMIT :top
    """
    df = pd.read_sql(q, con, params={"as_of": as_of, "surface": surface, "since": since, "cutoff": cutoff,
                                     "min_matches": min_matches, "min_recent": min_recent,
                                     "min_surface": min_surface, "top": top})
    df.insert(0, "rank", range(1, len(df) + 1))
    return df

for s in ["Clay", "Hard", "Grass"]:
    print(s)
    print(snapshot_surface("2024-12-22", s, top=10).to_string(index=False))
    print()

In [ ]:
def player_profile(name, as_of="2024-12-22"):
    q = """
    WITH ranked AS (
        SELECT e.surface, e.result, e.surf_offset_after,
               ROW_NUMBER() OVER (PARTITION BY e.surface ORDER BY e.match_id DESC) AS rn
        FROM elo_long e
        JOIN players p ON p.player_id = e.player_id
        WHERE p.name = :name AND e.tourney_start <= :as_of
    )
    SELECT surface,
           COUNT(*) AS matches,
           ROUND(100.0 * SUM(result) / COUNT(*), 1) AS win_pct,
           ROUND(MAX(CASE WHEN rn = 1 THEN surf_offset_after END)) AS offset_now
    FROM ranked
    GROUP BY surface
    ORDER BY matches DESC
    """
    return pd.read_sql(q, con, params={"name": name, "as_of": as_of})

for n in ["Jannik Sinner", "Carlos Alcaraz", "Rafael Nadal"]:
    print(n)
    print(player_profile(n).to_string(index=False))
    print()

In [ ]:
def ranking_rows(as_of, surface, top=20):
    if surface == "Overall":
        d = snapshot(as_of, min_matches=15, min_recent=15, top=top)
        return pd.DataFrame({
            "as_of": as_of, "surface": "Overall", "rank": d["rank"], "name": d["name"],
            "rating": d["elo"], "overall_rating": d["elo"], "surface_offset": np.nan,
            "matches_52w": d["matches_52w"], "surface_matches": np.nan, "note": d["note"]})
    d = snapshot_surface(as_of, surface, min_matches=15, min_recent=15, min_surface=20, top=top)
    return pd.DataFrame({
        "as_of": as_of, "surface": surface, "rank": d["rank"], "name": d["name"],
        "rating": d["elo_surface"], "overall_rating": d["overall"], "surface_offset": d["surf_offset"],
        "matches_52w": d["matches_52w"], "surface_matches": d["surface_matches"], "note": ""})


frames = []
for y in range(2015, 2026):
    frames.append(ranking_rows(f"{y}-12-22", "Overall"))
    frames.append(ranking_rows(f"{y}-12-22", "Hard"))
for y in range(2015, 2027):
    frames.append(ranking_rows(f"{y}-06-15", "Clay"))
    frames.append(ranking_rows(f"{y}-07-20", "Grass"))
latest = pd.read_sql("SELECT MAX(tourney_start) AS d FROM elo_long", con).iloc[0, 0]
frames.append(ranking_rows(latest, "Overall"))
frames.append(ranking_rows(latest, "Hard"))

tbl = pd.concat(frames, ignore_index=True)
tbl["source"] = "Calculated from TennisMyLife match data"
tbl["licence"] = "Derived aggregate; non-commercial, see data/Sources.md"
tbl["method"] = ("Elo v2 (overall rating + surface offset, calibrated). Eligibility: at least 15 matches in total and "
                 "15 in the last 52 weeks, last tournament within 182 days; surface lists need 20 surface matches. "
                 "Latest-date rows include tournaments in progress.")
tbl["computed_on"] = pd.Timestamp.today().strftime("%Y-%m-%d")
tbl.to_sql("elo_ranking_snapshots", con, if_exists="replace", index=False)
con.commit()

print("latest date:", latest)
print(tbl.groupby("surface").size().to_string())

In [ ]:
latest = pd.read_sql("SELECT MAX(tourney_start) AS d FROM elo_long", con).iloc[0, 0]

p = period_view("2026-01-01", latest, min_in_period=10, top=15).copy()
p["as_of"] = latest
p["source"] = "Calculated from TennisMyLife match data"
p["licence"] = "Derived aggregate; non-commercial, see data/Sources.md"
p["method"] = ("Elo v2 overall rating (surface offsets not included). Eligibility: at least 15 matches in total, "
               "10 in 2026, last tournament within 182 days. Latest date includes tournaments in progress.")
p["computed_on"] = pd.Timestamp.today().strftime("%Y-%m-%d")
p.to_sql("elo_period_2026", con, if_exists="replace", index=False)
con.commit()

print("as of:", latest)
print(p[["rank", "name", "elo_start", "elo_end", "change", "matches_in_period"]].to_string(index=False))

In [ ]:
def season_card(year, min_matches=30, top=15, sort_by="avg_elo"):
    p = {"y": year, "mm": min_matches}
    base = pd.read_sql("""
        SELECT e.player_id,
               COUNT(*) AS matches,
               SUM(e.result) AS wins,
               ROUND(AVG(e.elo_before)) AS avg_elo,
               ROUND(MAX(e.elo_after)) AS peak_elo,
               ROUND(AVG(o.elo_before)) AS avg_opp_elo,
               SUM(CASE WHEN e.expected < 0.5 THEN 1 ELSE 0 END) AS matches_as_underdog,
               SUM(CASE WHEN e.expected < 0.5 AND e.result = 1 THEN 1 ELSE 0 END) AS wins_as_underdog,
               ROUND(SUM(e.result - e.expected), 1) AS wins_above_expected
        FROM elo_long e
        JOIN elo_long o ON o.match_id = e.match_id AND o.player_id = e.opponent_id
        JOIN matches m ON m.match_id = e.match_id
        WHERE m.file_year = :y
        GROUP BY e.player_id
        HAVING COUNT(*) >= :mm
    """, con, params=p)

    titles = pd.read_sql("SELECT winner_id AS player_id, COUNT(*) AS titles FROM matches "
                         "WHERE round = 'F' AND file_year = :y GROUP BY winner_id", con, params={"y": year})
    lost = pd.read_sql("SELECT loser_id AS player_id, COUNT(*) AS finals_lost FROM matches "
                       "WHERE round = 'F' AND file_year = :y GROUP BY loser_id", con, params={"y": year})
    srv = pd.read_sql("SELECT server_id AS player_id, "
                      "ROUND(100.0 * SUM(sv_won) / SUM(svpt), 1) AS svc_pts_won_pct, "
                      "ROUND(100.0 * SUM(ace) / SUM(svpt), 1) AS aces_pct "
                      "FROM serve_obs WHERE file_year = :y GROUP BY server_id", con, params={"y": year})
    ret = pd.read_sql("SELECT returner_id AS player_id, "
                      "ROUND(100.0 * SUM(svpt - sv_won) / SUM(svpt), 1) AS ret_pts_won_pct "
                      "FROM serve_obs WHERE file_year = :y GROUP BY returner_id", con, params={"y": year})
    names = pd.read_sql("SELECT player_id, name FROM players", con)

    d = base
    for t in (titles, lost, srv, ret, names):
        d = d.merge(t, on="player_id", how="left")
    d[["titles", "finals_lost"]] = d[["titles", "finals_lost"]].fillna(0).astype(int)

    z = 1.96
    n, pw = d["matches"], d["wins"] / d["matches"]
    center = (pw + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(pw * (1 - pw) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    d["win_pct"] = (100 * pw).round(1)
    d["win_pct_low"] = (100 * (center - half)).round(1)
    d["win_pct_high"] = (100 * (center + half)).round(1)

    cols = ["name", "matches", "wins", "win_pct", "win_pct_low", "win_pct_high", "titles", "finals_lost",
            "avg_elo", "peak_elo", "avg_opp_elo", "wins_as_underdog", "matches_as_underdog",
            "wins_above_expected", "svc_pts_won_pct", "ret_pts_won_pct", "aces_pct"]
    return d.sort_values(sort_by, ascending=False).head(top)[cols]


print(season_card(2024).to_string(index=False))